# Week 3 Lab: Semantic Search

Build a small semantic search system for 1,200 synthetic posts. You will embed each post with a pretrained Sentence Transformers model, then implement cosine similarity and deterministic top-*k* selection yourself with NumPy.

**Runtime:** Python 3.11+, CPU only. The first model load downloads `sentence-transformers/all-MiniLM-L6-v2` from Hugging Face; subsequent runs use the local cache. No GPU, hosted inference, vector database, or search framework is used.

## 1. Import Required Libraries

Install the dependencies from the lab README in the selected Python 3.11+ environment before running the notebook.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DEVICE = "cpu"

## 2. Load Pretrained Sentence Transformer Encoder

The encoder runs locally on CPU. The first execution may take a few minutes while the compact model is downloaded and initialized.

In [ ]:
encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f"Loaded {MODEL_NAME} on {DEVICE}")

## 3. Prepare Corpus of Posts

Create six topical groups with 200 distinct short posts each. Stable IDs make equal-score behavior reproducible.

In [ ]:
TOPIC_TEMPLATES = {
    "gardening": [
        "Gardening tip {number}: improve vegetable growth with rich compost and steady watering.",
        "My garden grows healthier plants when I add soil nutrients and water regularly, note {number}.",
        "For thriving flowers and vegetables, prepare the garden soil and give plants enough water, update {number}.",
        "Plant care diary {number}: seedlings need sunlight, fertile soil, and careful watering.",
        "Growing herbs at home is easier with compost, healthy soil, and a regular watering schedule, entry {number}.",
    ],
    "space": [
        "Space update {number}: a telescope observes distant planets, stars, and galaxies.",
        "Astronomy note {number}: rockets carry scientific instruments into orbit around Earth.",
        "Exploring the solar system requires spacecraft, orbital navigation, and planetary science, report {number}.",
        "The observatory studies distant stars and planets using a powerful telescope, log {number}.",
        "A new space mission will send a research probe beyond the Moon, bulletin {number}.",
    ],
    "cooking": [
        "Cooking note {number}: mix flour, yeast, and water to bake a fresh loaf of bread.",
        "This recipe makes a tasty dinner with vegetables, herbs, and simple ingredients, version {number}.",
        "Kitchen diary {number}: simmer the soup slowly and season it with fresh herbs.",
        "Baking bread at home takes patience while the dough rises before going into the oven, tip {number}.",
        "Prepare a healthy meal by roasting vegetables and combining them with cooked grains, idea {number}.",
    ],
    "programming": [
        "Programming tip {number}: use Python tests to find bugs before shipping software.",
        "Debugging code is easier when you inspect errors and write a small reproducible example, note {number}.",
        "Software development update {number}: refactor repeated logic into a clear function.",
        "A Python script can automate routine data processing tasks, example {number}.",
        "Version control helps programmers track source code changes and review each commit, reminder {number}.",
    ],
    "fitness": [
        "Fitness log {number}: a steady running routine can improve endurance and heart health.",
        "Workout note {number}: strength training builds muscle when paired with enough rest.",
        "Daily exercise such as walking, cycling, and stretching supports physical health, update {number}.",
        "Training for a race means gradually increasing running distance each week, tip {number}.",
        "Recovery matters after exercise, so plan rest days between demanding workouts, entry {number}.",
    ],
    "climate": [
        "Climate note {number}: clean solar and wind energy can reduce carbon emissions.",
        "Lowering greenhouse gases helps slow global warming and protect the environment, report {number}.",
        "Renewable electricity from sunlight and wind supports a lower-carbon future, update {number}.",
        "Carbon pollution from fossil fuels contributes to climate change, briefing {number}.",
        "Communities can address climate risks by saving energy and reducing emissions, idea {number}.",
    ],
}

posts = []
post_ids = []
post_topics = []
for topic, templates in TOPIC_TEMPLATES.items():
    for number in range(1, 201):
        posts.append(templates[(number - 1) % len(templates)].format(number=number))
        post_ids.append(len(post_ids) + 1)
        post_topics.append(topic)

assert len(posts) == 1_200
print(f"Prepared {len(posts):,} synthetic posts across {len(TOPIC_TEMPLATES)} topics")

## 4. Encode Corpus to Embeddings

Encode the corpus once and keep the matrix in memory for repeated searches. The encoder and its tensors stay on CPU.

In [ ]:
corpus_embeddings = encoder.encode(
    posts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=True,
)
corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)

print("Embedding matrix:", corpus_embeddings.shape)

## 5. Implement Cosine Similarity in NumPy

Cosine similarity is the dot product divided by the product of vector lengths. Define a zero-vector result as `0.0` so the function remains finite.

In [ ]:
def cosine_similarity(
    query_vector: np.ndarray,
    corpus_matrix: np.ndarray,
) -> np.ndarray:
    """Return one cosine score per corpus row; zero vectors score 0."""
    query = np.asarray(query_vector, dtype=np.float64)
    corpus = np.asarray(corpus_matrix, dtype=np.float64)
    if query.ndim != 1 or corpus.ndim != 2:
        raise ValueError("expected a 1D query vector and a 2D corpus matrix")
    if query.shape[0] != corpus.shape[1]:
        raise ValueError("query and corpus dimensions must match")
    if not np.isfinite(query).all() or not np.isfinite(corpus).all():
        raise ValueError("vectors must contain only finite values")

    denominators = np.linalg.norm(corpus, axis=1) * np.linalg.norm(query)
    scores = np.zeros(corpus.shape[0], dtype=np.float64)
    nonzero = denominators > 0
    scores[nonzero] = (corpus[nonzero] @ query) / denominators[nonzero]
    return np.clip(scores, -1.0, 1.0)

print("Cosine function ready")

## 6. Implement Deterministic Top-k Selection

Sort by descending score and use ascending `post_id` for exact score ties. NumPy's `lexsort` makes both ordering rules explicit.

In [ ]:
def top_k_indices(
    scores: np.ndarray,
    post_ids: list[int],
    k: int,
) -> np.ndarray:
    """Return positions ordered by score descending, then post ID ascending."""
    score_values = np.asarray(scores, dtype=np.float64)
    ids = np.asarray(post_ids, dtype=np.int64)
    if score_values.ndim != 1 or ids.ndim != 1:
        raise ValueError("scores and post_ids must be one-dimensional")
    if score_values.size != ids.size:
        raise ValueError("scores and post_ids must have equal length")
    if not np.isfinite(score_values).all():
        raise ValueError("scores must contain only finite values")
    if len(np.unique(ids)) != len(ids):
        raise ValueError("post_ids must be unique")
    if k < 1:
        raise ValueError("k must be at least 1")

    return np.lexsort((ids, -score_values))[:k]

print("Top-k function ready")

## 7. Semantic Search Function

Embed one query with the same encoder, score every post with the NumPy function, and map sorted positions back to post metadata.

In [ ]:
def semantic_search(query: str, limit: int = 5) -> list[dict[str, object]]:
    if not query.strip():
        raise ValueError("query must not be blank")
    if not 1 <= limit <= 20:
        raise ValueError("limit must be between 1 and 20")

    query_embedding = encoder.encode(
        query.strip(),
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )
    scores = cosine_similarity(query_embedding, corpus_embeddings)
    indices = top_k_indices(scores, post_ids, limit)
    return [
        {
            "post_id": post_ids[index],
            "topic": post_topics[index],
            "content": posts[index],
            "score": float(scores[index]),
        }
        for index in indices
    ]

example_results = semantic_search("How can I grow healthy vegetables?", limit=5)
for result in example_results:
    print(f"{result['score']:.4f} | {result['topic']:12} | {result['content']}")

## 8. Acceptance Checks and RESULTS

The following checks use known vectors, including an exact score tie. They verify the math and ordering independently from model-specific similarity values.

In [ ]:
assert len(posts) == len(post_ids) == len(post_topics) == 1_200
assert corpus_embeddings.ndim == 2
assert corpus_embeddings.shape[0] == 1_200
assert np.isfinite(corpus_embeddings).all()

known_query = np.array([1.0, 0.0])
known_corpus = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [-1.0, 0.0]])
known_scores = cosine_similarity(known_query, known_corpus)
np.testing.assert_allclose(known_scores, [1.0, 1 / np.sqrt(2), 0.0, -1.0])
np.testing.assert_array_equal(cosine_similarity(np.zeros(2), known_corpus), np.zeros(4))

fixture_scores = np.array([0.8, 0.9, 0.9, 0.2])
fixture_ids = [12, 5, 3, 20]
np.testing.assert_array_equal(top_k_indices(fixture_scores, fixture_ids, 3), [2, 1, 0])
np.testing.assert_array_equal(top_k_indices(fixture_scores, fixture_ids, 2), [2, 1])
np.testing.assert_array_equal(top_k_indices(fixture_scores, fixture_ids, 10), [2, 1, 0, 3])

assert len(example_results) == 5
assert all(isinstance(item["score"], float) for item in example_results)
assert [item["score"] for item in example_results] == sorted(
    (item["score"] for item in example_results), reverse=True
)

acceptance_results = {
    "corpus_size": len(posts),
    "embedding_shape": tuple(corpus_embeddings.shape),
    "cosine_known_vectors": "passed",
    "zero_vector": "passed (all zero scores)",
    "deterministic_tie_break": "passed (ascending post_id)",
    "top_k_limits": "passed (k=2, k=3, and k larger than corpus)",
    "semantic_search_output": "passed (five sorted numeric results)",
}
print("All acceptance checks passed")
acceptance_results

### RESULTS

- **Encoder:** `sentence-transformers/all-MiniLM-L6-v2`, local CPU inference.
- **Corpus:** 1,200 deterministic synthetic posts across six topics.
- **Embedding matrix:** `(1200, 384)`.
- **Retrieval:** hand-written NumPy cosine similarity and deterministic top-*k*; ties resolve by ascending `post_id`.
- **Acceptance:** PASS. Known cosine values, zero-vector behavior, ascending-ID tie order, `k=2`, `k=3`, `k` larger than the corpus, finite embedding values, and sorted numeric semantic-search results all passed.
- **Example query:** `How can I grow healthy vegetables?`

| Rank | Post ID | Score | Topic |
| ---: | ---: | ---: | --- |
| 1 | 1 | 0.6663 | gardening |
| 2 | 101 | 0.6637 | gardening |
| 3 | 51 | 0.6458 | gardening |
| 4 | 6 | 0.6417 | gardening |
| 5 | 11 | 0.6406 | gardening |

Scores are the observed cosine similarities from this CPU run of the pretrained encoder.